# Notebook 2 of 3: poster figures and significance tests (163 original Northeast wells)

**Needs:** the three CSVs from notebook 1 (`cumulative_gaps_annual_163.csv`, `cumulative_gaps_seasonal_163.csv`, `seasonal_shares_163.csv`), `well_variables_163.csv` from notebook 3 (only the depth column is used), the DTW event tables, and the shapefiles / raw series listed in cell 1.

In [1]:
# ===== CELL 1 =====
%matplotlib qt
import os, time
import numpy as np, pandas as pd, geopandas as gpd, requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Polygon, Patch
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from shapely.geometry import box

SAVE_DIR = r"C:\Users\romin"                                       

# ---- inputs ----
EVENT_DIR         = "dtw_event_tables_rpsy"
GAPS_ANNUAL_CSV   = "cumulative_gaps_annual_163.csv"
GAPS_SEASONAL_CSV = "cumulative_gaps_seasonal_163.csv"
SHARES_CSV        = "seasonal_shares_163.csv"
VARIABLES_CSV     = "well_variables_163.csv"                        

# ---- shapefiles and raw series ----
PHYSIO_PATH  = r"C:\Users\romin\Downloads\physio_shp\physio.shp"
STATES_PATH  = r"C:\Users\romin\OneDrive\Groundwater\cb_2022_us_state_500k"
RECHARGE_DIR = "C:/Users/romin/OneDrive/Groundwater/RpSy Data"      
DAYMET_DIR   = "./daymet"                                           

STORY_WELL = "420828076484601"                                      
MIN_STORM  = 5                                                      

plt.rcParams.update({"font.size": 14, "axes.labelsize": 15, "axes.titlesize": 15, "xtick.labelsize": 13,
                     "ytick.labelsize": 13, "legend.fontsize": 12, "font.family": "DejaVu Sans"})
SEASONS = ["Winter", "Spring", "Summer", "Fall"]
COLORS  = {"Winter": "#4682B4", "Spring": "#2E8B57", "Summer": "#FF8C00", "Fall": "#B22222"}
MONTH2SEASON = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
                6: "Summer", 7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall"}
os.makedirs(SAVE_DIR, exist_ok=True)

def save_png(fig, name, **kw):
    path = os.path.join(SAVE_DIR, name)
    try:
        fig.savefig(path, dpi=300, **kw)
    except OSError:
        path = path.replace(".png", f"_{time.strftime('%H%M%S')}.png"); fig.savefig(path, dpi=300, **kw)
    print("saved:", path)

def save_csv(df, name, **kw):
    path = os.path.join(SAVE_DIR, name)
    try:
        df.to_csv(path, **kw)
    except OSError:
        path = path.replace(".csv", f"_{time.strftime('%H%M%S')}.csv"); df.to_csv(path, **kw)
    print("saved:", path)

# ---- the 163 wells and their mean depth to water ----
W = pd.read_csv(VARIABLES_CSV, dtype={"well_id": str})
if "original_163" in W.columns:
    W = W[W["original_163"].astype(str).str.lower().isin(["true", "1", "1.0"])]
W = W.drop_duplicates("well_id").set_index("well_id")
ids = set(W.index)

# ---- distance from the 1:1 line (notebook 1): gap_25 = distance at 25% of cumulative precipitation, and so on ----
annual   = pd.read_csv(GAPS_ANNUAL_CSV, dtype={"well_id": str}); annual = annual[annual["well_id"].isin(ids)].set_index("well_id")
seasonal = pd.read_csv(GAPS_SEASONAL_CSV, dtype={"well_id": str}); seasonal = seasonal[seasonal["well_id"].isin(ids)]
shares   = pd.read_csv(SHARES_CSV, dtype={"well_id": str}); shares = shares[shares["well_id"].isin(ids)]
GCOLS = [c for c in annual.columns if c.startswith("gap_")]
GRID  = np.array([int(c.split("_")[1]) / 100 for c in GCOLS])
gcol  = lambda mark: f"gap_{int(round(mark * 100)):02d}"
print(f"{len(W)} wells | annual curves: {annual['gap_50'].notna().sum()} | well-seasons: {len(seasonal)} | depth available: {W['mean_depth_to_water_m'].notna().sum()}")

163 wells | annual curves: 163 | well-seasons: 652 | depth available: 163


In [2]:
# ===== FIGURE 1: MAP OF THE 163 WELLS, COLORED BY DISTANCE FROM THE 1:1 LINE AT THE 50% MARK (annual curve) =====
MARK      = 0.50
HIGHLIGHT = STORY_WELL                  
PAD       = 0.5
PAL = {"NEW ENGLAND": "#F4A6C0", "ST. LAWRENCE VALLEY": "#FFD54F", "ADIRONDACK": "#B39DDB", "COASTAL PLAIN": "#D9BE9A",
       "APPALACHIAN PLATEAUS": "#F4A259", "VALLEY AND RIDGE": "#9ACD5F", "PIEDMONT": "#E57373", "BLUE RIDGE": "#64B5F6",
       "CENTRAL LOWLAND": "#80CBC4"}
NE = ["Connecticut", "Maine", "Massachusetts", "New Hampshire", "Rhode Island", "Vermont", "New Jersey", "New York", "Pennsylvania"]

m = W[["lat", "lon"]].join(annual[gcol(MARK)].rename("gap")).dropna().reset_index()
states = gpd.read_file(STATES_PATH)
states = states if states.crs is not None else states.set_crs(epsg=4326)
states = states[~states["NAME"].isin(["Alaska", "Hawaii", "Puerto Rico", "United States Virgin Islands", "Guam",
                                      "American Samoa", "Commonwealth of the Northern Mariana Islands"])].to_crs("EPSG:4326")
physio = gpd.read_file(PHYSIO_PATH)
physio = physio if physio.crs is not None else physio.set_crs(epsg=4326)
phys = physio.cx[-85:-64, 36:49]
phys = phys[phys["PROVINCE"].str.strip() != ""].to_crs("EPSG:5070").dissolve(by="PROVINCE", as_index=False)
phys["geometry"] = phys.buffer(3000).buffer(-3000)                      
phys = phys.to_crs("EPSG:4326")
phys["key"] = phys["PROVINCE"].str.strip().str.upper()
phys["color"] = phys["key"].map(PAL).fillna("#BDBDBD")

g = gpd.GeoDataFrame(m, geometry=gpd.points_from_xy(m["lon"], m["lat"]), crs="EPSG:4326")
sx0, sy0, sx1, sy1 = states[states["NAME"].isin(NE)].total_bounds
wx0, wy0, wx1, wy1 = g.total_bounds
x0, x1 = min(sx0, wx0) - PAD, max(sx1, wx1) + PAD
y0, y1 = min(sy0, wy0) - PAD, max(sy1, wy1) + PAD

fig, ax = plt.subplots(figsize=(11, 6.2), layout="constrained")
phys.plot(ax=ax, color=phys["color"], edgecolor="white", linewidth=0.8, zorder=1)
states.boundary.plot(ax=ax, edgecolor="#4d4d4d", linewidth=0.6, zorder=3)
lim = float(np.nanpercentile(np.abs(g["gap"]), 95))
sc = ax.scatter(g.geometry.x, g.geometry.y, c=g["gap"], s=52, edgecolor="black", linewidth=0.6, zorder=5,
                cmap="RdBu", norm=TwoSlopeNorm(vcenter=0, vmin=-lim, vmax=lim))
cb = fig.colorbar(sc, ax=ax, location="right", shrink=0.6, aspect=30, extend="both")
cb.set_label(f"Distance from 1:1 line ({int(MARK * 100)}% mark)")

handles = []
if HIGHLIGHT is not None and str(HIGHLIGHT) in set(g["well_id"]):
    h = g[g["well_id"] == str(HIGHLIGHT)].iloc[0]
    ax.scatter([h.geometry.x], [h.geometry.y], s=330, facecolor="none", edgecolor="black", linewidth=2.2, zorder=6)
    handles.append(Line2D([], [], marker="o", ms=13, mfc="none", mec="black", mew=2.2, ls="", label=f"Well {HIGHLIGHT}"))

ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
ax.set_aspect(1 / np.cos(np.deg2rad((y0 + y1) / 2))); ax.set_axis_off()          # north straight up
ax.set_title(f"Northeast wells (n = {len(g)})")
present = set(phys.loc[phys.intersects(box(x0, y0, x1, y1)), "key"])
keys = [k for k in PAL if k in present] + sorted(k for k in present if k not in PAL)
handles = [mpatches.Patch(facecolor=PAL.get(k, "#BDBDBD"), label=k.title().replace(" And ", " and ")) for k in keys] + handles
fig.legend(handles=handles, title="Physiographic province", loc="outside left center", frameon=False, alignment="left")
save_png(fig, f"fig1_map_distance_{int(MARK * 100)}_163.png")
plt.show()

saved: C:\Users\romin\fig1_map_distance_50_163.png


In [3]:
# ===== FIGURE 2: WATER TABLE FLUCTUATION (WTF) METHOD WITH A MASTER RECESSION CURVE =====
WELL        = STORY_WELL
PEAK_DATE   = None                     
START, END  = "2000-01-01", time.strftime("%Y-%m-%d")
PRE_DAYS, POST_DAYS = 20, 30          
LOOKBACK    = 7                       
SLOPE_LENS  = [4, 6, 8, 10]            
MIN_GAPS    = [2, 4, 6, 8]            
RAIN_MM     = 0.5                      
RED = "#B22222"

# ---------- 1. daily depth to water from USGS  ----------
r = requests.get("https://waterservices.usgs.gov/nwis/dv/", timeout=60,
                 params=dict(format="json", sites=WELL, parameterCd="72019", startDT=START, endDT=END, siteStatus="all"))
r.raise_for_status()
series = [s for s in r.json()["value"]["timeSeries"] if s["values"] and s["values"][0]["value"]]
if not series: raise ValueError(f"USGS returned no daily depth-to-water values for {WELL} (parameter 72019).")
s = max(series, key=lambda z: len(z["values"][0]["value"]))
nodata = float(s["variable"].get("noDataValue", -999999))
raw = pd.DataFrame(s["values"][0]["value"])
raw["date"] = pd.to_datetime(raw["dateTime"]).dt.tz_localize(None).dt.normalize()
raw["v"] = pd.to_numeric(raw["value"], errors="coerce").replace(nodata, np.nan)
depth = (raw.groupby("date")["v"].mean() * (0.3048 if s["variable"]["unit"]["unitCode"].lower().startswith("ft") else 1.0)).asfreq("D")
H = -depth                                                                  

# ---------- 2. precipitation (Daymet) and the dataset's RpSy ----------
P = None
try:
    dm = pd.read_csv(f"{DAYMET_DIR}/{WELL}.csv")
    dm.index = pd.to_datetime(dm["year"].astype(int).astype(str), format="%Y") + pd.to_timedelta(dm["yday"].astype(int) - 1, unit="D")
    P = dm["prcp (mm/day)"].reindex(H.index)
except Exception as e: print("Daymet not used (recession picked from water levels only):", e)
RP = None
try: RP = pd.read_csv(f"{RECHARGE_DIR}/{WELL}.csv", parse_dates=["Date"]).set_index("Date")["RpSy (m)"]
except Exception as e: print("RpSy file not used:", e)

# ---------- 3. master recession curve---------
best = None
for L in SLOPE_LENS:
    rate = (H.shift(-L) - H) / L                                            
    Hm = H.rolling(L + 1).mean().shift(-L)                                  
    for gap in MIN_GAPS:
        if P is not None:
            ok = ((P >= RAIN_MM).astype(float).rolling(L + gap + 1).sum().shift(-L) == 0)     
        else:
            ok = (H.diff().rolling(L).max().shift(-L) <= 0)                  
        e = ok & rate.notna() & Hm.notna() & (rate <= 0)
        if e.sum() < 30: continue
        lr = stats.linregress(Hm[e], rate[e]); n = int(e.sum())
        adj = 1 - (1 - lr.rvalue ** 2) * (n - 1) / (n - 2)
        if best is None or adj > best["adj"]:
            best = dict(L=L, gap=gap, adj=adj, a=lr.intercept, b=lr.slope, n=n)
if best is None: raise RuntimeError("Not enough rain-free recession periods to fit a master recession curve.")
print(f"MRC: slope element {best['L']} d, min rain-free gap {best['gap']} d, n = {best['n']}, adj-R2 = {best['adj']:.2f}, "
      f"dH/dt = {best['a']:.4f} + ({best['b']:.4f}) x H")
if best["b"] >= 0: print("WARNING: the fitted MRC does not slope downward with H; this well may not suit the method.")
mrc = lambda Hh: best["a"] + best["b"] * Hh                               

# ---------- 4. the rise to draw ----------
if PEAK_DATE is None:
    rise = H - H.rolling(LOOKBACK, min_periods=max(3, LOOKBACK // 2)).min().shift(1)
    peak = rise.idxmax()
else: peak = pd.Timestamp(PEAK_DATE)
t0 = H.loc[peak - pd.Timedelta(days=LOOKBACK):peak - pd.Timedelta(days=1)].dropna().idxmin()
w0, w1 = t0 - pd.Timedelta(days=PRE_DAYS), peak + pd.Timedelta(days=POST_DAYS)
days = pd.date_range(t0, peak)
print(f"rise {t0.date()} to {peak.date()}")

# ---------- 5. figure ----------
fig, (ax, axb) = plt.subplots(2, 1, figsize=(9, 7), sharex=True, gridspec_kw={"height_ratios": [3, 1.2]}, layout="constrained")
ax.plot(depth.loc[w0:w1].index, depth.loc[w0:w1].values, color="black", lw=2.2, zorder=3, label="Observed depth to water")
first = True
for d in days[:-1]:                                                        
    d1 = d + pd.Timedelta(days=1)
    if pd.isna(H.get(d)) or pd.isna(H.get(d1)): continue
    pred = -(H[d] + mrc(H[d]))
    ax.plot([d, d1], [depth[d], pred], color=RED, ls="--", lw=2.2, zorder=4, label="Expected level with no recharge (master recession curve)" if first else None)
    ax.plot([d1, d1], [pred, depth[d1]], color=RED, lw=3, zorder=4, label="\u0394H: daily rise above the recession" if first else None)
    first = False
ax.invert_yaxis(); ax.set_ylabel("Depth to water (m)"); ax.grid(alpha=0.3)
ax.set_title(f"Water table fluctuation method, well {WELL}")
if RP is not None:                                                         
    axb.plot(RP.loc[w0:w1].index, RP.loc[w0:w1].values, color=RED, lw=1.4)
    axb.fill_between(RP.loc[t0:peak].index, 0, RP.loc[t0:peak].values, color=RED, alpha=0.25)
    axb.set_ylabel("RpSy (m)"); axb.set_ylim(bottom=0); axb.grid(alpha=0.3)
else: axb.set_visible(False)
axb.set_xlim(w0, w1)
axb.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=5)); axb.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%Y"))
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower left", ncol=1, frameon=False, fontsize=13, alignment="left")
save_png(fig, f"fig2_wtf_mrc_method_{WELL}.png")
plt.show()

MRC: slope element 4 d, min rain-free gap 8 d, n = 1036, adj-R2 = 0.42, dH/dt = -0.2717 + (-0.0364) x H
rise 2011-09-02 to 2011-09-08
saved: C:\Users\romin\fig2_wtf_mrc_method_420828076484601.png


In [4]:
# ===== FIGURE 3: STORY WELL TIME SERIES (a) AND DTW-PAIRED ZOOM (b) =====
well       = STORY_WELL
ZOOM_START = "1989-03-19"         
ZOOM_DAYS  = 60
LONG_DAYS  = 4                   
MIN_STORMS, MAX_STORMS = 4, 8     
FIGSIZE    = (8, 8.4)             
RED, BLUE  = "#d62728", "#1f77b4"

# ---- data ----
dcols = ["precip_start_date", "precip_end_date", "recharge_start_date", "recharge_end_date"]
ev = pd.read_csv(f"{EVENT_DIR}/{well}_dtw_event_table.csv", parse_dates=dcols)
ev = ev[ev["precip_total"] >= MIN_STORM].dropna(subset=["recharge_start_date"]).copy()        
ev["storm_days"] = (ev["precip_end_date"] - ev["precip_start_date"]).dt.days + 1
ev["rech_days"] = (ev["recharge_end_date"] - ev["recharge_start_date"]).dt.days + 1
ev["span"] = ev[["storm_days", "rech_days"]].max(axis=1)

rr = pd.read_csv(f"{RECHARGE_DIR}/{well}.csv", parse_dates=["Date"]).set_index("Date")
pp = pd.read_csv(f"{DAYMET_DIR}/{well}.csv")
pp["Date"] = pd.to_datetime(pp["year"] * 1000 + pp["yday"], format="%Y%j"); pp = pp.set_index("Date")
df = rr[["RpSy (m)"]].join(pp["prcp (mm/day)"], how="left")
ok = df["RpSy (m)"].dropna().index
df = df.loc[ok.min():ok.max()]                                   
print(f"Record: {ok.min().date()} to {ok.max().date()}")

# ---- zoom window ----
one = pd.Timedelta(days=1)
if ZOOM_START:
    z0 = pd.Timestamp(ZOOM_START.strip())
else:
    rows = []
    for s0 in pd.date_range(ok.min(), ok.max() - pd.Timedelta(days=ZOOM_DAYS), freq="3D"):
        s1 = s0 + pd.Timedelta(days=ZOOM_DAYS)
        inw = ev[(ev["precip_start_date"] >= s0) & (ev["recharge_end_date"] < s1)]
        if len(inw) == 0 or df.loc[s0:s1].isna().any().any(): continue
        rows.append(dict(start=s0, n=len(inw), longest=int(inw["span"].max()), recharge=df.loc[s0:s1, "RpSy (m)"].sum()))
    cand = pd.DataFrame(rows)
    good = cand[cand["n"].between(MIN_STORMS, MAX_STORMS) & (cand["longest"] >= LONG_DAYS)]
    if good.empty:
        print("No window met the rules; using the best available."); good = cand[cand["n"] <= MAX_STORMS]
    good = good.sort_values(["longest", "recharge"], ascending=False)
    shown = []
    for _, c in good.iterrows():
        if all(abs((c["start"] - s["start"]).days) > 30 for s in shown): shown.append(c)
        if len(shown) == 5: break
    print("Top windows (set ZOOM_START to any of these):")
    for c in shown: print(f"  {c['start'].date()} | {int(c['n'])} storms | longest pairing {int(c['longest'])} days")
    z0 = shown[0]["start"]
z1 = z0 + pd.Timedelta(days=ZOOM_DAYS)
win = df.loc[z0:z1]
print(f"Zoom window: {z0.date()} to {z1.date()}")

# ---- figure ----
fig, (axT, axB) = plt.subplots(2, 1, figsize=FIGSIZE, layout="constrained")

def panel(ax, d, lw):
    ax2 = ax.twinx()
    ax.plot(d.index, d["RpSy (m)"], color=RED, lw=lw)
    ax2.plot(d.index, d["prcp (mm/day)"], color=BLUE, lw=lw)
    ax.set_ylim(0, d["RpSy (m)"].max() * 1.7); ax2.set_ylim(d["prcp (mm/day)"].max() * 1.7, 0)    
    ax.set_ylabel("RpSy (m)", color=RED); ax2.set_ylabel("Precipitation (mm)", color=BLUE)
    ax.tick_params(axis="y", colors=RED); ax2.tick_params(axis="y", colors=BLUE)
    ax.set_xlim(d.index[0], d.index[-1]); ax.set_xlabel("Date")

def date_ticks(ax, loc):
    ax.xaxis.set_major_locator(loc); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%Y"))

panel(axT, df, 0.8)                                             
date_ticks(axT, mdates.YearLocator(4) if (df.index[-1] - df.index[0]).days > 365 * 15 else mdates.YearLocator(2))
axT.axvspan(z0, z1, color="0.5", alpha=0.35, lw=0, zorder=0)
axT.set_title(f"(a) Well {well}: full record")

panel(axB, win, 1.2)                                            
date_ticks(axB, mdates.DayLocator(interval=14))
inwin = ev[(ev["precip_start_date"] >= z0) & (ev["recharge_end_date"] < z1)]
for _, e in inwin.iterrows():
    t0, t1 = mdates.date2num(e["precip_start_date"]), mdates.date2num(e["precip_end_date"] + one)
    b0, b1 = mdates.date2num(e["recharge_start_date"]), mdates.date2num(e["recharge_end_date"] + one)
    axB.add_patch(Polygon([(t0, 1), (t1, 1), (b1, 0), (b0, 0)], closed=True, transform=axB.get_xaxis_transform(),
                          facecolor="0.55", edgecolor="0.3", linewidth=0.8, alpha=0.55, zorder=0))
axB.set_title("(b) Zoom: shaded = DTW pairing of each storm with its recharge")
print(f"{len(inwin)} storms in the zoom window")
if len(inwin):
    L = inwin.loc[inwin["span"].idxmax()]
    print(f"Longest pairing in the window: storm {L['precip_start_date'].date()} to {L['precip_end_date'].date()} ({int(L['storm_days'])} d)"
          f" -> recharge {L['recharge_start_date'].date()} to {L['recharge_end_date'].date()} ({int(L['rech_days'])} d)")
save_png(fig, f"fig3_timeseries_and_dtw_zoom_{well}.png")
plt.show()

Record: 1988-10-02 to 2022-09-30
Zoom window: 1989-03-19 to 1989-05-18
8 storms in the zoom window
Longest pairing in the window: storm 1989-03-28 to 1989-04-05 (9 d) -> recharge 1989-03-29 to 1989-04-05 (8 d)
saved: C:\Users\romin\fig3_timeseries_and_dtw_zoom_420828076484601.png


In [5]:
# ===== FIGURE 4: STORY WELL CUMULATIVE CURVE (a) AND THE SAME CURVE BY SEASON (b) =====
well   = STORY_WELL
MARKS  = [0.25, 0.50, 0.75]
FIGSIZE = (13.5, 6.6)
BLUE, CRIMSON = "steelblue", "crimson"

e = pd.read_csv(f"{EVENT_DIR}/{well}_dtw_event_table.csv", usecols=["precip_start_date", "precip_total", "recharge_total"], parse_dates=["precip_start_date"])
e["recharge_total"] = e["recharge_total"].fillna(0)                       
e = e[e["precip_total"] >= MIN_STORM].copy()
e["season"] = e["precip_start_date"].dt.month.map(MONTH2SEASON)

def curve(d):
    """cumulative fraction of precipitation (x) vs cumulative fraction of recharge (y), storms smallest -> largest"""
    d = d.sort_values("precip_total")
    x = np.concatenate(([0], (d["precip_total"].cumsum() / d["precip_total"].sum()).values))
    y = np.concatenate(([0], (d["recharge_total"].cumsum() / d["recharge_total"].sum()).values))
    return x, y

TICKS = [0, 0.25, 0.5, 0.75, 1]; TLAB = ["0", "0.25", "0.5", "0.75", "1"]
def square(ax):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_box_aspect(1)
    ax.set_xticks(TICKS); ax.set_xticklabels(TLAB); ax.set_yticks(TICKS); ax.set_yticklabels(TLAB)

fig = plt.figure(figsize=FIGSIZE, layout="constrained")
gs = fig.add_gridspec(2, 4)
axO = fig.add_subplot(gs[:, 0:2])                                         
axS = [fig.add_subplot(gs[0, 2]), fig.add_subplot(gs[0, 3]), fig.add_subplot(gs[1, 2]), fig.add_subplot(gs[1, 3])]   

x, y = curve(e)
axO.plot([0, 1], [0, 1], "k--", lw=1.2, zorder=1, label="1:1 line")
axO.plot(x, y, color=BLUE, lw=3, zorder=3, label="Cumulative curve")
gaps_story = []
for m in MARKS:                                                           
    yc = float(np.interp(m, x, y)); g = yc - m; gaps_story.append(g)
    axO.plot([m, m], [m, yc], color=CRIMSON, lw=2.8, zorder=4)
    axO.scatter([m], [m], s=70, facecolor="white", edgecolor="black", linewidth=1.5, zorder=5)
    axO.scatter([m], [yc], s=70, facecolor=BLUE, edgecolor="black", linewidth=1.5, zorder=5)
    axO.annotate(f"{g:.3f}", (m, (m + yc) / 2), xytext=(-8, 10), textcoords="offset points", ha="right", va="bottom",
                 color=CRIMSON, fontsize=13, fontweight="bold")
square(axO)
axO.set_xlabel("Cumulative precipitation fraction"); axO.set_ylabel("Cumulative recharge fraction")
axO.set_title("(a) All storms"); axO.legend(loc="upper left")

for ax, sea in zip(axS, SEASONS):
    d = e[e["season"] == sea]
    xs, ys = curve(d)
    ax.plot([0, 1], [0, 1], ls="--", color="0.5", lw=1.6, zorder=1)
    ax.plot(xs, ys, color="black", lw=1.4, marker="o", ms=2.2, mec="black", zorder=3)
    square(ax); ax.grid(alpha=0.25)
    ax.set_xlabel("Cum. precipitation"); ax.set_ylabel("Cum. recharge"); ax.set_title(f"{sea} (n = {len(d)})")
axS[0].set_title(f"(b) Winter (n = {int((e['season'] == 'Winter').sum())})")
fig.suptitle(f"Well {well}")
print("Distance from the 1:1 line at 25/50/75%:", ", ".join(f"{g:.3f}" for g in gaps_story), "| storms per season:", {s: int((e['season'] == s).sum()) for s in SEASONS})
save_png(fig, f"fig4_cumulative_curves_{well}.png")
plt.show()

Distance from the 1:1 line at 25/50/75%: -0.046, -0.084, -0.062 | storms per season: {'Winter': 269, 'Spring': 367, 'Summer': 426, 'Fall': 352}
saved: C:\Users\romin\fig4_cumulative_curves_420828076484601.png


In [6]:
# ===== FIGURE 5: SEASONAL SHARES OF RECHARGE AND PRECIPITATION (a) + DISTANCE FROM THE 1:1 LINE AT 25 / 50 / 75 % BY SEASON (b) =====
MARKS   = [0.25, 0.50, 0.75]
FIGSIZE = (16, 5)                                  # one row, every panel the same square size
FLIER   = dict(marker="o", markersize=3.5, markerfacecolor="none", markeredgecolor="black")

def box(ax, vals, pos, width, color):
    bp = ax.boxplot(vals, positions=[pos], widths=width, patch_artist=True, medianprops=dict(color="black"),
                    boxprops=dict(edgecolor="black"), flierprops=FLIER)
    bp["boxes"][0].set_facecolor(color)

fig, axs = plt.subplots(1, 4, figsize=FIGSIZE, layout="constrained")

ax = axs[0]                                      
for i, sea in enumerate(SEASONS):
    d = shares[shares["season"] == sea]
    box(ax, d["recharge_share"], i - 0.18, 0.3, COLORS[sea])
    box(ax, d["precip_share"], i + 0.18, 0.3, "white")
ax.axhline(0.25, color="black", ls="--", lw=1)
ax.set_xticks(range(4)); ax.set_xticklabels(SEASONS, fontsize=11)
ax.set_ylabel("Share of the well's annual total"); ax.set_title("(a) Overall")
lo, hi = ax.get_ylim(); ax.set_ylim(lo, hi + 0.18 * (hi - lo))              
ax.legend(handles=[Patch(facecolor="0.7", edgecolor="black", label="Recharge (colored)"),
                   Patch(facecolor="white", edgecolor="black", label="Precipitation (white)")],
          loc="upper center", fontsize=10.5, frameon=False)

vals = {m: seasonal[["well_id", "season", gcol(m)]].rename(columns={gcol(m): "gap"}) for m in MARKS}     
lim = max(np.abs(vals[m]["gap"]).max() for m in MARKS) * 1.05
for k, (ax, m) in enumerate(zip(axs[1:], MARKS)):
    for i, sea in enumerate(SEASONS):
        box(ax, vals[m].loc[vals[m]["season"] == sea, "gap"].dropna(), i, 0.55, COLORS[sea])
    ax.axhline(0, color="black", ls="--", lw=1)
    ax.set_ylim(-lim, lim); ax.set_xticks(range(4)); ax.set_xticklabels(SEASONS, fontsize=11)
    ax.set_title(f"(b) {int(round(m * 100))}% point" if k == 0 else f"{int(round(m * 100))}% point")
    if k == 0: ax.set_ylabel("Distance from 1:1 line")
    else:      ax.tick_params(axis="y", labelleft=False)
for ax in axs: ax.set_box_aspect(1)                                        

print("Median share of annual total (%):")
print((shares.groupby("season")[["recharge_share", "precip_share"]].median().reindex(SEASONS) * 100).round(1).to_string())
save_png(fig, "fig5_season_shares_and_distance_163.png")
plt.show()

Median share of annual total (%):
        recharge_share  precip_share
season                              
Winter            28.6          21.0
Spring            29.3          24.2
Summer            17.5          29.0
Fall              23.2          26.6
saved: C:\Users\romin\fig5_season_shares_and_distance_163.png


In [7]:
# ===== FIGURE 6: MEDIAN DISTANCE FROM THE 1:1 LINE ALONG THE WHOLE CURVE, WINTER vs SUMMER (band = middle 50% of wells) =====
A, B = "Summer", "Winter"
FIGSIZE = (8, 6)

xg = np.concatenate(([0], GRID, [1]))                                     
cols = {}
for sea in (A, B):
    t = seasonal[seasonal["season"] == sea].set_index("well_id")[GCOLS]
    cols[sea] = t
both = cols[A].index.intersection(cols[B].index)                          
fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")
for sea in (B, A):
    M = cols[sea].loc[both].to_numpy()
    M = np.column_stack([np.zeros(len(M)), M, np.zeros(len(M))])
    ax.fill_between(xg, np.percentile(M, 25, axis=0), np.percentile(M, 75, axis=0), color=COLORS[sea], alpha=0.2)
    ax.plot(xg, np.median(M, axis=0), color=COLORS[sea], lw=3, label=f"{sea} median")
ax.axhline(0, color="black", ls="--", lw=1.2, label="1:1 line")
ax.set_xlim(0, 1); ax.grid(alpha=0.3)
ax.set_xlabel("Cumulative fraction of precipitation (small to large storms)")
ax.set_ylabel("Distance from the 1:1 line")
ax.set_title(f"{len(both)} wells")
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=3, frameon=False)
save_png(fig, "fig6_median_distance_winter_vs_summer_163.png")
plt.show()

saved: C:\Users\romin\fig6_median_distance_winter_vs_summer_163.png


In [8]:
# ===== FIGURE 7: DEPTH TO WATER vs DISTANCE FROM THE 1:1 LINE AT THE 50% MARK, BINNED BY 20% OF WELLS =====
MARK   = 0.50
N_BINS = 5                                        
FIGSIZE = (8, 6)

depth_w = W["mean_depth_to_water_m"].dropna()                             
d = seasonal[["well_id", "season", gcol(MARK)]].rename(columns={gcol(MARK): "gap"}).dropna()
d["depth"] = d["well_id"].map(depth_w)
d = d.dropna(subset=["depth"])

wb = pd.qcut(depth_w.rank(method="first"), N_BINS, labels=False)           
d["bin"] = d["well_id"].map(wb)
d = d.dropna(subset=["bin"]); d["bin"] = d["bin"].astype(int)
b = d.groupby("bin").agg(n_wells=("well_id", "nunique"), n=("gap", "size"),
                         depth_min=("depth", "min"), depth_med=("depth", "median"), depth_max=("depth", "max"),
                         q25=("gap", lambda v: v.quantile(.25)), med=("gap", "median"), q75=("gap", lambda v: v.quantile(.75))).reset_index()
edges = [(b.loc[i, "depth_max"] + b.loc[i + 1, "depth_min"]) / 2 for i in range(len(b) - 1)]     

fig, ax = plt.subplots(figsize=FIGSIZE, layout="constrained")
for s in SEASONS:
    q = d[d["season"] == s]
    ax.scatter(q["depth"], q["gap"], s=34, color=COLORS[s], edgecolor="black", linewidth=0.4, alpha=0.45, zorder=1, label=f"{s} (n = {len(q)})")
ax.axhline(0, color="black", ls="--", lw=1.2, zorder=2, label="1:1 line")
ax.errorbar(b["depth_med"], b["med"], yerr=[b["med"] - b["q25"], b["q75"] - b["med"]], fmt="o-", color="black", mfc="white", mew=1.4,
            ms=10, lw=1.8, capsize=4, elinewidth=2, zorder=4, label=f"Median and IQR of each bin")
for x_ in edges: ax.axvline(x_, color="0.35", ls=":", lw=1.0, zorder=2)
ax.set_xlim(left=0); ax.grid(alpha=0.3)
ax.set_xlabel("Mean depth to water (m)"); ax.set_ylabel(f"Distance from 1:1 line ({int(MARK * 100)}% mark)")
ax.set_title(f"Each bin = {int(100 / N_BINS)}% of wells ({b['n_wells'].min()}-{b['n_wells'].max()} wells per bin)")
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=3, frameon=False, fontsize=11)
save_png(fig, f"fig7_depth_vs_distance_{int(MARK * 100)}_bins{N_BINS}_163.png")
save_csv(b.round(3), f"fig7_depth_bins_{int(MARK * 100)}_163.csv", index=False)
print(b.round(3).to_string(index=False))
plt.show()

saved: C:\Users\romin\fig7_depth_vs_distance_50_bins5_163.png
saved: C:\Users\romin\fig7_depth_bins_50_163.csv
 bin  n_wells   n  depth_min  depth_med  depth_max    q25    med   q75
   0       33 132      0.461      1.526      1.899 -0.062 -0.008 0.037
   1       32 128      1.899      2.396      2.792 -0.110 -0.052 0.026
   2       33 132      2.817      3.561      4.593 -0.091 -0.006 0.059
   3       32 128      4.689      5.692      7.110 -0.067  0.016 0.088
   4       33 132      7.186     10.633     40.548 -0.068  0.014 0.124


In [10]:
# ===== TEST 1: SPEARMAN RANK CORRELATION, MEAN DEPTH TO WATER vs DISTANCE FROM THE 1:1 LINE (Fig 7 and Results) =====

MARKS = [0.25, 0.50, 0.75]
depth_w = W["mean_depth_to_water_m"].dropna()

rows = []
a = annual[[gcol(m) for m in MARKS]].join(depth_w.rename("depth"), how="inner").dropna()
for m in MARKS:
    rho, p = stats.spearmanr(a["depth"], a[gcol(m)])
    rows.append(dict(curve="Annual (all storms)", mark=f"{int(m * 100)}%", n=len(a), rho=rho, p=p))
for sea in SEASONS:
    for m in MARKS:
        t = seasonal.loc[seasonal["season"] == sea, ["well_id", gcol(m)]].dropna()
        t = t.assign(depth=t["well_id"].map(depth_w)).dropna()
        rho, p = stats.spearmanr(t["depth"], t[gcol(m)])
        rows.append(dict(curve=sea, mark=f"{int(m * 100)}%", n=len(t), rho=rho, p=p))
sp = pd.DataFrame(rows)
sp["p_bonferroni"] = (sp["p"] * len(sp)).clip(upper=1)                      
out = sp.copy(); out["rho"] = out["rho"].round(2); out["p"] = out["p"].map(lambda v: f"{v:.2g}"); out["p_bonferroni"] = out["p_bonferroni"].map(lambda v: f"{v:.2g}")
print(out.to_string(index=False))
print("\nPoster: annual rho = 0.31 (25%), 0.24 (50%, p = 0.0017), 0.17 (75%); weaker within seasons, weakest in summer.")
save_csv(sp, "test_spearman_depth_vs_distance_163.csv", index=False)

              curve mark   n  rho       p p_bonferroni
Annual (all storms)  25% 163 0.30 8.3e-05       0.0012
Annual (all storms)  50% 163 0.24  0.0023        0.034
Annual (all storms)  75% 163 0.16   0.039         0.58
             Winter  25% 163 0.19   0.015         0.22
             Winter  50% 163 0.16   0.046         0.69
             Winter  75% 163 0.11    0.15            1
             Spring  25% 163 0.28 0.00023       0.0035
             Spring  50% 163 0.22  0.0056        0.083
             Spring  75% 163 0.13    0.11            1
             Summer  25% 163 0.20  0.0096         0.14
             Summer  50% 163 0.09    0.27            1
             Summer  75% 163 0.06    0.47            1
               Fall  25% 163 0.23  0.0031        0.047
               Fall  50% 163 0.16   0.044         0.66
               Fall  75% 163 0.06    0.42            1

Poster: annual rho = 0.31 (25%), 0.24 (50%, p = 0.0017), 0.17 (75%); weaker within seasons, weakest in summer.
saved: C

In [11]:
# ===== TEST 2: GLOBAL MORAN'S I (is the pattern spatially clustered?) for distance from the 1:1 line at 50% and for mean depth to water =====
# Weights = 1 / great-circle distance (diagonal 0). Wells closer than MERGE_KM are merged into one site (single linkage, values averaged).
# I is near its expected value -1/(N-1) when there is no clustering; the p-value comes from shuffling the values N_PERM times.
MARK     = 0.50
MERGE_KM = 1.0
N_PERM   = 9999
SEED     = 42

d = W[["lat", "lon", "mean_depth_to_water_m"]].join(annual[gcol(MARK)].rename("gap")).rename(columns={"mean_depth_to_water_m": "depth"})
d = d.dropna(subset=["gap", "depth", "lat", "lon"]).reset_index()

def dist_km(lat, lon):                                                       
    la, lo = np.radians(lat), np.radians(lon)
    h = np.sin((la[:, None] - la[None, :]) / 2) ** 2 + np.cos(la)[:, None] * np.cos(la)[None, :] * np.sin((lo[:, None] - lo[None, :]) / 2) ** 2
    D = 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))
    np.fill_diagonal(D, 0)
    return D

D0 = dist_km(d["lat"].values, d["lon"].values)                                
d["site"] = fcluster(linkage(squareform(D0, checks=False), "single"), t=MERGE_KM, criterion="distance")
site = d.groupby("site").agg(gap=("gap", "mean"), depth=("depth", "mean"), lat=("lat", "mean"), lon=("lon", "mean")).reset_index()
N = len(site)
print(f"{len(d)} wells -> {N} sites")

Ds = dist_km(site["lat"].values, site["lon"].values)
Wt = np.where(Ds > 0, 1.0 / np.where(Ds > 0, Ds, 1), 0.0)

def morans_i(v):
    z = v - v.mean()
    return (N / Wt.sum()) * (z @ Wt @ z) / (z @ z)

rng = np.random.default_rng(SEED)
perms = np.argsort(rng.random((N_PERM, N)), axis=1)
rows = []
for name, col in [(f"Distance from 1:1 line at the {int(MARK * 100)}% mark", "gap"), ("Mean depth to water (m)", "depth")]:
    v = site[col].values
    I = morans_i(v)
    z = v - v.mean(); zp = z[perms]
    Ip = (N / Wt.sum()) * np.einsum("ij,ij->i", zp @ Wt, zp) / (z @ z)         
    p = (1 + (np.abs(Ip - Ip.mean()) >= abs(I - Ip.mean())).sum()) / (N_PERM + 1)  
    rows.append(dict(variable=name, morans_I=round(I, 3), expected=round(-1 / (N - 1), 3),
                     z_score=round((I - Ip.mean()) / Ip.std(), 2), p_perm=round(p, 4)))
mor = pd.DataFrame(rows)
print(mor.to_string(index=False))
print("\nPoster: Moran's I = 0.068 (distance at 50%) and 0.038 (mean depth to water).")
save_csv(mor, f"test_morans_i_gap{int(MARK * 100)}_and_depth_163.csv", index=False)

163 wells -> 158 sites
                              variable  morans_I  expected  z_score  p_perm
Distance from 1:1 line at the 50% mark     0.068    -0.006     4.68  0.0004
               Mean depth to water (m)     0.037    -0.006     2.79  0.0154

Poster: Moran's I = 0.068 (distance at 50%) and 0.038 (mean depth to water).
saved: C:\Users\romin\test_morans_i_gap50_and_depth_163.csv


In [12]:
# ===== TEST 3 (supports the Discussion sentence on depth): is the depth link explained by how long recharge lasts, or by unpaired storms? =====
# Prints tables only (no figure). Uses the pairing statistics notebook 1 saved in cumulative_gaps_annual_163.csv.
MARK = 0.50
keep = [c for c in annual.columns if not c.startswith("gap_")] + [gcol(MARK)]
w = annual[keep].join(W["mean_depth_to_water_m"].rename("depth"), how="inner")
w["gap"] = w[gcol(MARK)]
w = w.dropna(subset=["depth"])
print(f"{len(w)} wells\n")

print(f"{'vs mean depth':30s}{'rho':>7s}{'p':>10s}{'n':>6s}")
for c, lab in [("frac_unpaired", "fraction of storms unpaired"), ("unpaired_small", "  unpaired, smallest third"),
               ("unpaired_large", "  unpaired, largest third"), ("med_storm_paired", "median size, paired (mm)"),
               ("med_storm_unpaired", "median size, unpaired (mm)"), ("med_lag_days", "median lag (days)"),
               ("med_recharge_days", "median recharge days"), ("gap", f"distance at {int(MARK * 100)}% mark")]:
    t = w[["depth", c]].dropna()
    r, p = stats.spearmanr(t["depth"], t[c]); print(f"{lab:30s}{r:7.2f}{p:10.3g}{len(t):6d}")

def partial(df, x, y, controls):
    """Rank partial correlation of x and y after removing what the control variables explain."""
    q = df[[x, y] + controls].dropna().rank()
    A = np.column_stack([np.ones(len(q))] + [q[c] for c in controls])
    res = lambda v: v - A @ np.linalg.lstsq(A, v, rcond=None)[0]
    r, _ = stats.pearsonr(res(q[x].values), res(q[y].values)); n = len(q)
    t = r * np.sqrt((n - 2 - len(controls)) / (1 - r ** 2)); p = 2 * stats.t.sf(abs(t), n - 2 - len(controls))
    return r, p, n

print("\nDepth vs distance from the 1:1 line, same test after controlling for:")
r, p = stats.spearmanr(*w[["depth", "gap"]].dropna().values.T); print(f"  {'nothing':32s} rho = {r:5.2f}  p = {p:.3g}")
for ctrl in (["frac_unpaired"], ["med_lag_days"], ["med_recharge_days"], ["frac_unpaired", "med_lag_days", "med_recharge_days"]):
    r, p, n = partial(w, "depth", "gap", ctrl); print(f"  {', '.join(ctrl):32s} rho = {r:5.2f}  p = {p:.3g}  (n = {n})")
t = w[["gap", "med_recharge_days"]].dropna(); r, p = stats.spearmanr(*t.values.T)
print(f"\nDistance vs median recharge days: rho = {r:.2f}  p = {p:.3g}")

163 wells

vs mean depth                     rho         p     n
fraction of storms unpaired      0.10     0.214   163
  unpaired, smallest third      -0.00     0.974   163
  unpaired, largest third        0.18    0.0196   163
median size, paired (mm)         0.01     0.863   163
median size, unpaired (mm)       0.23   0.00267   162
median lag (days)                0.22   0.00427   163
median recharge days             0.18    0.0195   163
distance at 50% mark             0.24   0.00227   163

Depth vs distance from the 1:1 line, same test after controlling for:
  nothing                          rho =  0.24  p = 0.00227
  frac_unpaired                    rho =  0.29  p = 0.000177  (n = 163)
  med_lag_days                     rho =  0.25  p = 0.00165  (n = 163)
  med_recharge_days                rho =  0.16  p = 0.0464  (n = 163)
  frac_unpaired, med_lag_days, med_recharge_days rho =  0.15  p = 0.0633  (n = 163)

Distance vs median recharge days: rho = 0.69  p = 5.82e-24


In [13]:
# =====  THE NUMBERS QUOTED IN THE POSTER TEXT, RECOMPUTED =====
rows = []
med = seasonal.groupby("season")[GCOLS].median()

rows.append(("Summer median distance at the 50% mark", "about -0.10", f"{med.loc['Summer', 'gap_50']:.3f}"))
sh = shares.groupby("season")[["recharge_share", "precip_share"]].median() * 100
rows.append(("Summer: share of annual recharge / precipitation", "about 18% / 29%", f"{sh.loc['Summer', 'recharge_share']:.0f}% / {sh.loc['Summer', 'precip_share']:.0f}%"))
rows.append(("Winter: share of annual recharge / precipitation", "about 29% / 21%", f"{sh.loc['Winter', 'recharge_share']:.0f}% / {sh.loc['Winter', 'precip_share']:.0f}%"))
rows.append(("Summer median curve below the 1:1 line at every point", "yes", "yes" if (med.loc["Summer"] < 0).all() else "no"))
rows.append(("Summer median curve furthest from the line at", "50%", f"{int(med.loc['Summer'].abs().idxmax().split('_')[1])}%"))
rows.append(("Winter median curve above the line", "yes", "yes" if (med.loc["Winter"] > 0).all() else "no"))
rows.append(("Winter median curve furthest from the line at", "about 25%", f"{int(med.loc['Winter'].abs().idxmax().split('_')[1])}%"))

# largest 10% of storms: share of recharge minus share of precipitation, per well, median across wells (storms >= MIN_STORM, unpaired = 0)
ex = {s: [] for s in SEASONS + ["Annual"]}
for wid in W.index:
    f = f"{EVENT_DIR}/{wid}_dtw_event_table.csv"
    if not os.path.exists(f): continue
    ev = pd.read_csv(f, parse_dates=["precip_start_date"], usecols=["precip_start_date", "precip_total", "recharge_total"])
    ev["recharge_total"] = ev["recharge_total"].fillna(0)
    ev = ev[ev["precip_total"] >= MIN_STORM].copy()
    ev["season"] = ev["precip_start_date"].dt.month.map(MONTH2SEASON)
    for key, e_ in [("Annual", ev)] + [(s, ev[ev["season"] == s]) for s in SEASONS]:
        if len(e_) < 20 or e_["recharge_total"].sum() == 0: continue
        e_ = e_.sort_values("precip_total"); k = max(1, int(np.ceil(0.10 * len(e_)))); top = e_.iloc[-k:]
        ex[key].append(top["recharge_total"].sum() / e_["recharge_total"].sum() - top["precip_total"].sum() / e_["precip_total"].sum())
rows.append(("Largest 10% of summer storms: recharge share minus rain share", "about +8 points", f"{np.median(ex['Summer']) * 100:+.1f} points"))

chk = pd.DataFrame(rows, columns=["statement on the poster", "poster says", "recomputed"])
print(chk.to_string(index=False))
print("\nLargest 10% of storms, median (recharge share - rain share) in points, by season:")
print({k: round(float(np.median(v)) * 100, 1) for k, v in ex.items() if v})
save_csv(chk, "poster_numbers_check_163.csv", index=False)

                                      statement on the poster     poster says  recomputed
                       Summer median distance at the 50% mark     about -0.10      -0.107
             Summer: share of annual recharge / precipitation about 18% / 29%   18% / 29%
             Winter: share of annual recharge / precipitation about 29% / 21%   29% / 21%
        Summer median curve below the 1:1 line at every point             yes         yes
                Summer median curve furthest from the line at             50%         51%
                           Winter median curve above the line             yes         yes
                Winter median curve furthest from the line at       about 25%         22%
Largest 10% of summer storms: recharge share minus rain share about +8 points +8.0 points

Largest 10% of storms, median (recharge share - rain share) in points, by season:
{'Winter': -1.9, 'Spring': -1.0, 'Summer': 8.0, 'Fall': 1.6, 'Annual': 0.4}
saved: C:\Users\romin\poster_nu